<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# Solution: Rules for patterns, and the price of each rule

**[Rules for patterns, and the price of each rule](https://learning.nextia-ai.com/courses/anomaly-detection/m03/rules-for-patterns/)** · Anomaly Detection: Find What Is Unusual · Module 3, lesson 2 of 7 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** add the Western Electric run rules to a control chart, and to measure the price of each rule with the average run length (ARL): how long until a false alarm, and how long until a real shift is caught.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells (less than 1 MB). No account. |
| **You should know** | The I-MR chart and its limits from phase I, from [Control charts: is this variation normal?](https://learning.nextia-ai.com/courses/anomaly-detection/m03/control-charts/). |
| **You do not need** | A statistics package: the rules and the simulation are a few lines of pandas and NumPy. |
| **Tested** | Python 3.12.13 in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*), with the package versions of the setup cell |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m03/rules-for-patterns/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M03-L02-rules-for-patterns/rules-for-patterns-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2. It needs an internet connection. The first time, it takes up to a minute.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. A restart is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib']:
    print(p, version(p))

## Setup: get the course data

Run the next cell. It downloads the data that this lesson uses (`refunds_daily.csv`, `sealed-key.txt`) into a folder `anomaly-course/data`, and checks each file's checksum. It also downloads `compare.py`, the course's shared code. It needs an internet connection and no account. It takes a few seconds.

You should see `Ready:` and the files with their sizes. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `anomaly-course` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course: no real people, cards or payments. [The dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/data/dataset-card.md) describes every table.

In [ ]:
# Setup: download the course data into anomaly-course/data/ and check each file.
# Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/"
FILES = {
    "refunds_daily.csv": "04c6384d1fbe8da908a739edc084eabc413581d75fce7cd84dcf7a90988996a6",
    "sealed-key.txt": "144cfd16737bdfcc4e992ad8ede0b8856265a2193c89fa7266fe0b27cb62ff98",
}
CODE_FILES = ["compare.py"]
HERE = Path.cwd()
# The notebook works in its own folder anomaly-course/, marked with a file, so it never writes into another project.
if Path(".nextia-notebook").exists():          # the cell ran before: start again from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("anomaly-course").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, else download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder anomaly-course and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, WORK / "data" / name, expected)
for name in CODE_FILES:
    fetch(name, LABS + name, WORK / name)
os.chdir(WORK)
print("Ready:", ", ".join(f"{n} ({(Path('data') / n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd`, NumPy as `np` and matplotlib as `plt`. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it, and the chart style.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple, set)):
            return [norm(x) for x in (sorted(v, key=str) if isinstance(v, set) else v)]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The chart in sigmas

The last lesson drew the refund rate with a centre line and limits 3 sigmas away. Here we write each day as **z**: how many sigmas it is from the centre line. A day at the upper limit has z = 3.

Run the next cell. It computes the limits of phase I again, and prints z for the first week of the planted shift (from 2025-04-07).

> **Check.** You should see a centre of 0.02137 and a sigma of 0.00621.

In [ ]:
import compare
daily = pd.read_csv("data/refunds_daily.csv", parse_dates=["date"]).set_index("date")
rate = daily.refund_rate.dropna()
phase1 = rate.loc["2024-04-01":"2024-12-31"]          # the stable reference period
center = phase1.mean()
sigma = phase1.diff().abs().mean() / 1.128
z = (rate - center) / sigma                            # how many sigmas each day is from the centre line
print(f"centre {center:.5f}, sigma {sigma:.5f}")
print(z.loc["2025-04-07":"2025-04-13"].round(2).to_string())

Each z is below 3, so the one-point rule says nothing. The shift adds about half a sigma to every day, and one day cannot show that. A run of many days on one side of the centre can.

## 2. The four Western Electric rules

The **Western Electric rules** were written at a telephone factory in 1956. Each rule looks for a pattern that is rare when nothing has changed:

1. One point beyond 3 sigma.
2. Two of three points in a row beyond 2 sigma, on the same side.
3. Four of five points in a row beyond 1 sigma, on the same side.
4. Eight points in a row on the same side of the centre line.

Run the next cell. It defines `fired(z, rule)`, which marks the days where one rule fires, and `alarms(z, rules)`, which combines rules. It prints nothing.

In [ ]:
RULES = {1: "1 point beyond 3 sigma", 2: "2 of 3 beyond 2 sigma",
         3: "4 of 5 beyond 1 sigma", 4: "8 in a row on one side"}

def fired(z, rule):
    """True on each day where one Western Electric rule fires, on the high side or the low side."""
    days = pd.Series(False, index=z.index)
    for s in (z, -z):                                  # the high side, then the low side
        if rule == 1:
            days |= s > 3
        if rule == 2:
            days |= (s > 2) & ((s > 2).astype(int).rolling(3, min_periods=1).sum() >= 2)
        if rule == 3:
            days |= (s > 1) & ((s > 1).astype(int).rolling(5, min_periods=1).sum() >= 4)
        if rule == 4:
            days |= (s > 0).astype(int).rolling(8).sum() == 8
    return days

def alarms(z, rules):
    """True on each day where at least one of the rules fires."""
    return pd.concat([fired(z, r) for r in rules], axis=1).any(axis=1)

## 3. The rules on twelve days

> **Predict.** Twelve days have these z values: 0.4, −0.6, 1.2, 0.5, 1.4, 1.1, 0.3, 1.6, 2.3, 0.8, 2.6 and 1.2. No day is beyond 3 sigma. Which rules fire, and on which days? Write your answer, then run the cell.

> **Check.** You should see the first `True` on day 9.

In [ ]:
tiny = pd.Series([0.4, -0.6, 1.2, 0.5, 1.4, 1.1, 0.3, 1.6, 2.3, 0.8, 2.6, 1.2],
                 index=pd.RangeIndex(1, 13, name="day"))      # 12 days, in sigmas from the centre
table = pd.DataFrame({"z": tiny} | {f"rule {r}": fired(tiny, r) for r in RULES})
print(table.to_string())

Rule 3 fires on day 9: four of the five days 5 to 9 are above 1 sigma. Rule 4 fires on day 10: days 3 to 10 are all above the centre. Rule 2 fires on day 11: days 9 and 11 are above 2 sigma. The one-point rule never fires.

## 4. The rules on Larkfield's refunds

Run the next cell. For four sets of rules, it gives the first alarm after the planted shift (2025-04-07) and the false alarms per month in the stable months before it (2025-01-01 to 2025-04-06).

In [ ]:
phase2 = z.loc["2025-01-01":]
rows = []
for rules in [(1,), (1, 4), (1, 2, 4), (1, 2, 3, 4)]:
    report = compare.alarm_report(phase2.index, alarms(phase2, rules), "2025-04-07", ("2025-01-01", "2025-04-06"))
    rows.append({"rules": " + ".join(map(str, rules)), **report})
real = pd.DataFrame(rows)
print(real.to_string(index=False))
stable = phase2.loc[:"2025-04-06"]
print("stable days where rule 4 fires:", [str(d.date()) for d in stable.index[fired(stable, 4)]])

> **Check.** You should see `2025-07-14` for rule 1 alone and `2025-06-11` with all four rules.

With all four rules, the chart found the shift 65 days after it started, against 98 days with rule 1 alone. But rule 4 also fired on 7 days of the stable months, in two runs of days below the centre: 2025-02-09, 2025-02-10, 2025-02-11, 2025-02-12, 2025-02-13, 2025-04-04, 2025-04-05. Nothing had changed, so these are false alarms. The table counts alarm **days**: a long run fires again on each new day. Also, one series is one story: one lucky or unlucky week changes it. To know the price of a rule, we need many series.

## 5. The average run length (ARL)

The **run length** is the number of days until the first alarm. Its average over many runs is the **average run length (ARL)**. Two ARLs describe a chart:

- **In control:** nothing changed, so the first alarm is a false alarm. We want this to be **long**.
- **After a shift:** the first alarm finds the change. We want this to be **short**.

We cannot rerun Larkfield's year 2,000 times, so we simulate it. Run the next cell. It makes 2,000 runs of 1,500 days with the same process as the data: the orders of phase I days, a day-to-day effect, and refunds drawn order by order. It takes a few seconds.

In [ ]:
orders_pool = daily.loc["2024-04-01":"2024-12-31", "orders"].to_numpy()
p1 = daily.loc["2024-04-01":"2024-12-31"]
pbar = p1.refunds.sum() / p1.orders.sum()            # the refund rate of phase I

def simulate(shift_points, runs=2000, days=1500):
    """runs x days of daily refund rates, made as Larkfield's process makes them, as z of the I-MR chart."""
    n = rng.choice(orders_pool, size=(runs, days))                     # orders: like the days of phase I
    p = (pbar + shift_points / 100) * np.exp(rng.normal(0, 0.10, (runs, days)))   # a day-to-day effect
    r = rng.binomial(n.astype(int), np.clip(p, 0, 1)) / n            # refunds: each order refunded or not
    return (r - center) / sigma

rng = np.random.default_rng(34)
z_normal = simulate(0.0)          # 2,000 runs of 1,500 days with nothing new
z_shift = simulate(0.3)           # the same process with the 0.3-point shift from day 1
print("simulated days:", z_normal.size + z_shift.size)

> **Predict.** Each rule adds a way to alarm. Which rule set will catch the shift first? And which gives the most false alarms? Write your answer, then run the cell.

In [ ]:
def count_last(b, w):
    """For each day, how many of the last w days (this day too) are True."""
    c = np.cumsum(b, axis=1)
    c[:, w:] = c[:, w:] - c[:, :-w]
    return c

def alarms_2d(z, rules):
    out = np.zeros(z.shape, bool)
    for s in (z, -z):
        if 1 in rules:
            out |= s > 3
        if 2 in rules:
            out |= (s > 2) & (count_last(s > 2, 3) >= 2)
        if 3 in rules:
            out |= (s > 1) & (count_last(s > 1, 5) >= 4)
        if 4 in rules:
            out |= count_last(s > 0, 8) >= 8
    return out

def run_length(a):
    """Days to the first alarm of each run (1,501 if a run has none)."""
    return np.where(a.any(axis=1), a.argmax(axis=1) + 1, a.shape[1] + 1)

def arl(rules):
    """Average run length: in control (a false alarm) and after the 0.3-point shift (a real alarm)."""
    normal, shifted = run_length(alarms_2d(z_normal, rules)).mean(), run_length(alarms_2d(z_shift, rules)).mean()
    return {"rules": " + ".join(map(str, rules)), "days to a false alarm": round(normal, 1),
            "days to catch the shift": round(shifted, 1), "false alarms a month": round(30.44 / normal, 2)}

price = pd.DataFrame([arl(r) for r in [(1,), (1, 4), (1, 2, 4), (1, 2, 3, 4)]])
print(price.to_string(index=False))

> **Check.** You should see 209.8 days to a false alarm with rule 1 alone, and 80.2 with all four rules.

Every rule makes the chart faster on the shift: from 60.6 days with rule 1 to 23.2 days with all four. And every rule makes false alarms more frequent: from one every 209.8 days to one every 80.2 days. This is the **price of a rule**. Choose the rules for the cost of a false alarm and the cost of a late alarm, not "as many as possible".

## 6. Check yourself

**Your turn 1.** On which day after the shift (from 2025-04-07) does rule 4 **alone** fire for the first time on Larkfield's data? Write it as a string `"YYYY-MM-DD"` in `first_rule4`.

In [ ]:
first_rule4 = str(phase2.index[fired(phase2, 4) & (phase2.index >= "2025-04-07")][0].date())

In [ ]:
expect_hash('first_rule4', first_rule4, '2eaaf20e31c069e3')

**Your turn 2.** Use `arl()` for rules 1 and 2 together. How many days to a false alarm? Write the number as `arl_rules_1_2`.

In [ ]:
arl_rules_1_2 = arl((1, 2))["days to a false alarm"]

In [ ]:
expect('arl_rules_1_2', arl_rules_1_2, 157.3, tolerance=0.001)

**Your turn 3.** A textbook 3-sigma chart on normal data has an in-control ARL of about 370 days. How many false alarms a month is that? Use 30.44 days a month, round to 2 decimals, and write it as `alarms_a_month`.

In [ ]:
alarms_a_month = round(30.44 / 370, 2)

In [ ]:
expect('alarms_a_month', alarms_a_month, 0.08, tolerance=0.001)

## Recap

- A run rule finds a pattern that no single point shows: eight days on one side of the centre, or four of five beyond 1 sigma.
- The average run length gives the price of a rule: rule 1 alone gave a false alarm every 209.8 days and caught the 0.3-point shift in 60.6 days; all four rules, 80.2 and 23.2 days.
- More rules, faster alarms and more false alarms. Choose the rules for the cost of each.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Charts for rates and counts](https://learning.nextia-ai.com/courses/anomaly-detection/m03/charts-for-rates-and-counts/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m03/rules-for-patterns/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.